# 🫁 Personalized Lung Disease Management System
## Full Training Pipeline — Google Colab T4 GPU

This notebook runs the complete pipeline:
1. Mount Google Drive & clone/upload project
2. Install dependencies
3. Configure Kaggle credentials
4. Download ICBHI 2017 + Arashnic datasets
5. Preprocess audio → spectrograms
6. Train model (Stage 1: 50 epochs + Stage 2: 15 epochs)
7. Evaluate model
8. Start API server + ngrok tunnel

**Estimated total time:** ~3–4 hours on T4 GPU

---
**Before running:** Go to `Runtime → Change runtime type → GPU (T4)`

## Step 1 — Check GPU

In [ ]:
!nvidia-smi
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Step 2 — Mount Google Drive (to persist checkpoints)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create a folder in Drive to store checkpoints and outputs
import os
DRIVE_DIR = '/content/drive/MyDrive/LungDiseaseModel'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive folder ready: {DRIVE_DIR}')

## Step 3 — Upload or clone the project

In [ ]:
import os

PROJECT_DIR = '/content/Lung_Disease_Management'

# OPTION A: If you have the project on GitHub, uncomment and edit:
!git clone https://github.com/Sarnika-anbu/Lung_Disease_Management.git {PROJECT_DIR}

# OPTION B: If uploading a zip — run this cell, then upload via the file browser
# After uploading lung_disease_management.zip, run:
# !unzip /content/lung_disease_management.zip -d /content/

# OPTION C: Already cloned / uploaded
# Just verify the directory exists:
if os.path.exists(PROJECT_DIR):
    print(f'Project found at {PROJECT_DIR}')
    !ls {PROJECT_DIR}
else:
    print('Project not found. Please upload or clone the project first.')
    print('Run: !git clone https://github.com/YOUR_USERNAME/Lung_Disease_Management.git', PROJECT_DIR)

os.chdir(PROJECT_DIR)
print(f'Working directory: {os.getcwd()}')

## Step 4 — Install dependencies

In [ ]:
# Core ML and audio packages (torch is pre-installed on Colab)
!pip install -q timm==0.9.16 grad-cam==1.5.0 librosa==0.10.2 noisereduce==3.0.2
!pip install -q reportlab==4.0.9 pypdf==4.2.0
!pip install -q fastapi==0.111.0 uvicorn==0.29.0 python-multipart==0.0.9
!pip install -q hypothesis==6.102.6 pytest==8.2.1
!pip install -q altair==5.3.0 pyngrok
print('All packages installed.')

## Step 5 — Configure Kaggle API credentials

In [ ]:
import os
import json

# PASTE YOUR KAGGLE USERNAME AND API KEY HERE
KAGGLE_USERNAME = 'YOUR_KAGGLE_USERNAME'  # <-- edit this
KAGGLE_KEY      = 'YOUR_KAGGLE_API_KEY'  # <-- edit this

# Write kaggle.json
kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)
kaggle_json_path = os.path.join(kaggle_dir, 'kaggle.json')
with open(kaggle_json_path, 'w') as f:
    json.dump({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}, f)
os.chmod(kaggle_json_path, 0o600)

print(f'Kaggle credentials written to {kaggle_json_path}')
print('Verifying...')
!kaggle datasets list --max-size 1 2>&1 | head -3

## Step 6 — Download datasets (~3.5 GB)

In [ ]:
import sys
sys.path.insert(0, '/content/Lung_Disease_Management')

# Download ICBHI 2017 and Arashnic datasets
!python src/data/download.py

# Verify
!echo '--- ICBHI directory ---'
!ls data/raw/icbhi/ | head -5
!echo '--- Arashnic directory ---'
!ls data/raw/arashnic/ | head -5

## Step 7 — Preprocess audio → spectrograms (30–60 min)

In [ ]:
# This preprocesses all audio files:
# - Resamples to 4000 Hz
# - Applies bandpass filter (50–2000 Hz) + spectral gating
# - Segments into 5-second windows
# - Computes 128-bin log-mel spectrograms + delta + delta-delta
# - Saves (3, 128, 216) .npy tensors
# - Creates train/test split CSVs
!python src/data/preprocess.py

# Verify splits
import pandas as pd
train_df = pd.read_csv('data/processed/splits/train.csv')
test_df  = pd.read_csv('data/processed/splits/test.csv')
print(f'Train samples: {len(train_df)}')
print(f'Test samples:  {len(test_df)}')
print(f'Train class distribution:\n{train_df["disease_class"].value_counts()}')

## Step 8 — Train the model (Stage 1: 50 epochs + Stage 2: 15 epochs)

In [ ]:
# This runs both training stages:
# Stage 1: 50 epochs, full EfficientNetV2B0+CBAM+metadata model
#   - FocalLoss (gamma=2, label_smoothing=0.1)
#   - WeightedRandomSampler for class imbalance
#   - SpecAugment + Mixup augmentation
#   - AdamW (lr=1e-4) + CosineAnnealingLR
# Stage 2: 15 epochs, frozen backbone, head fine-tuning
#   - Class-balanced subset

!python src/training/train.py

# Copy checkpoint to Drive for persistence
import shutil
if os.path.exists('checkpoints/best.pth'):
    shutil.copy('checkpoints/best.pth', f'{DRIVE_DIR}/best.pth')
    print(f'Checkpoint saved to {DRIVE_DIR}/best.pth')

## Step 9 — Evaluate the model

In [ ]:
!python src/training/evaluate.py

# Display the output plots
from IPython.display import Image, display
import os

if os.path.exists('outputs/confusion_matrix.png'):
    print('Confusion Matrix:')
    display(Image('outputs/confusion_matrix.png'))

if os.path.exists('outputs/reliability_diagram.png'):
    print('Reliability Diagram:')
    display(Image('outputs/reliability_diagram.png'))

# Copy outputs to Drive
shutil.copy('outputs/confusion_matrix.png', f'{DRIVE_DIR}/confusion_matrix.png')
shutil.copy('outputs/reliability_diagram.png', f'{DRIVE_DIR}/reliability_diagram.png')
print('Evaluation outputs saved to Drive.')

## Step 10 — Start the API server with ngrok tunnel

In [ ]:
# Install and configure ngrok to expose the API over the internet
# Get a free ngrok token from: https://dashboard.ngrok.com/get-started/your-authtoken

NGROK_TOKEN = 'YOUR_NGROK_AUTHTOKEN'  # <-- edit this

from pyngrok import ngrok, conf
conf.get_default().auth_token = NGROK_TOKEN

# Kill any existing ngrok tunnels
ngrok.kill()

# Start uvicorn in the background
import subprocess, time
api_proc = subprocess.Popen(
    ['uvicorn', 'src.api.main:app', '--host', '0.0.0.0', '--port', '8000'],
    cwd='/content/Lung_Disease_Management'
)
time.sleep(3)  # Wait for server to start

# Open ngrok tunnel
tunnel = ngrok.connect(8000)
API_URL = tunnel.public_url
print(f'🚀 API is live at: {API_URL}')
print(f'   Health check:  {API_URL}/health')
print(f'   API docs:      {API_URL}/docs')

## Step 11 — Test the API

In [ ]:
import requests

# Health check
r = requests.get(f'{API_URL}/health')
print('Health:', r.json())

# Test prediction with a sample WAV file
# Replace with a real WAV file path from the test set
import pandas as pd
test_df = pd.read_csv('data/processed/splits/test.csv')

# Find a real audio file from the raw data
import glob
wav_files = glob.glob('data/raw/icbhi/**/*.wav', recursive=True)
if wav_files:
    sample_wav = wav_files[0]
    print(f'Testing with: {sample_wav}')
    with open(sample_wav, 'rb') as f:
        response = requests.post(
            f'{API_URL}/predict',
            files={'audio_file': ('test.wav', f, 'audio/wav')},
            data={
                'age': '65',
                'sex': 'M',
                'bmi': '26.5',
                'smoking_pack_years': '15',
                'recording_location': 'Trachea'
            }
        )
    print('Prediction:', response.json())
else:
    print('No WAV files found. Run preprocessing first.')

## Step 12 — Use the Streamlit Frontend

The Streamlit app can't run directly in Colab (it requires a browser). Instead:

**Option A — Run locally** (point to the Colab API URL):
```bash
# On your local machine:
API_BASE_URL=https://xxxx.ngrok.io streamlit run app/streamlit_app.py
```

**Option B — Use ngrok with Streamlit in Colab** (run the cell below)

In [ ]:
# Run Streamlit in Colab with a tunnel
import os
os.environ['API_BASE_URL'] = API_URL  # Point frontend to the Colab API

# Start Streamlit
st_proc = subprocess.Popen(
    ['streamlit', 'run', 'app/streamlit_app.py',
     '--server.port', '8501', '--server.headless', 'true'],
    cwd='/content/Lung_Disease_Management'
)
time.sleep(5)

# Open a tunnel for Streamlit
st_tunnel = ngrok.connect(8501)
print(f'🖥️  Streamlit frontend: {st_tunnel.public_url}')
print('Open the link above in your browser.')

## Step 13 — Download checkpoint and outputs

Your checkpoint is already saved to Google Drive (`LungDiseaseModel/best.pth`).  
Run the cell below to also download it directly to your computer.

In [ ]:
from google.colab import files

# Download the best model checkpoint
if os.path.exists('checkpoints/best.pth'):
    files.download('checkpoints/best.pth')
    print('Downloading best.pth...')
else:
    print('No checkpoint found. Run training first (Step 8).')

# Download evaluation outputs
for f_path in ['outputs/confusion_matrix.png', 'outputs/reliability_diagram.png']:
    if os.path.exists(f_path):
        files.download(f_path)

---
## Notes

| Metric | Target | Notes |
|--------|--------|-------|
| ICBHI Score | ≥ 0.90 | (sensitivity + specificity) / 2, macro-averaged |
| Inference time | < 3 s | classification + Grad-CAM |
| GPU memory | < 15 GB | batch_size=32 |

**If training is interrupted** (Colab disconnects), the best checkpoint is saved to Google Drive.  
Restart from Step 8 — the dataset preprocessing only needs to run once.

**To use the trained model locally:**
1. Download `best.pth` from Drive
2. Place it in `checkpoints/best.pth`
3. Run `uvicorn src.api.main:app --port 8000`
4. Run `streamlit run app/streamlit_app.py`